# GroundingDINO + SAM 2 자동 주석 초안과 CVAT 검수

이 노트북은 AIHub 후보 이미지 500장에 접시 전체(`plate_full`)와 보이는 음식(`food_visible`) 초안을 만들고, CVAT에서 **검수만** 하도록 준비합니다. GPU 런타임을 사용하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline')
assert PROJECT_ROOT.is_dir(), f'프로젝트 경로를 확인하세요: {PROJECT_ROOT}'
%cd {PROJECT_ROOT}

In [ ]:
# 기존 코랩 PyTorch를 유지하고 프로젝트의 호환 의존성만 보완합니다.
!pip install --prefer-binary --upgrade-strategy only-if-needed -r requirements-colab.txt
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'

In [ ]:
# 자동 주석보다 먼저, 정확히 같은 500장과 작업 목록 CSV를 준비합니다.
from pathlib import Path
WORK_ROOT = Path('data/training/plate_segmentation')
manifest = WORK_ROOT / 'plate_annotation_manifest.csv'
images_dir = WORK_ROOT / 'cvat_images'
RESET_PLATE_WORKSPACE = False  # 이전 53장 등 불완전 작업 폴더를 새 500장으로 교체할 때만 True
if not manifest.exists():
    if images_dir.exists() and any(images_dir.iterdir()) and not RESET_PLATE_WORKSPACE:
        raise RuntimeError(
            'cvat_images는 있지만 plate_annotation_manifest.csv가 없습니다. '
            '기존 결과를 백업한 뒤 이 셀의 RESET_PLATE_WORKSPACE=True로 바꾸고 다시 실행하세요.'
        )
    reset_arg = '--reset-output' if RESET_PLATE_WORKSPACE else ''
    !python -m scripts.prepare_plate_annotation_manifest --sample-size 500 {reset_arg}
image_files = [p for p in images_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
assert manifest.is_file(), f'작업 목록 생성 실패: {manifest}'
assert len(image_files) == 500, f'입력 이미지는 500장이어야 합니다. 현재: {len(image_files)}장'
print('작업 목록과 입력 이미지 확인 완료:', len(image_files), '장')

In [ ]:
# 첫 실행은 GroundingDINO와 SAM 2.1 Small 가중치를 내려받으므로 시간이 걸립니다.
# 원본 GroundingDINO/SAM2 저장소를 별도 설치하지 마세요. 현재 프로젝트 어댑터를 사용합니다.
!python -m scripts.generate_plate_segmentation_drafts --overwrite

In [ ]:
import json
from IPython.display import Image, display
summary = json.loads(Path('data/training/plate_segmentation/auto_annotations/draft_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
previews = sorted(Path(summary['preview_dir']).glob('*.jpg'))
if previews:
    display(Image(filename=str(previews[0])))
print('CVAT 업로드 파일:', summary['coco_json'])

## CVAT 검수 뒤

`cvat_images`를 CVAT Task에 올린 뒤 `instances_draft.json`을 COCO Instances 형식으로 가져옵니다. 현재처럼 일부 검수본만 먼저 학습할 때는 검수한 범위만 필터링한 COCO 파일을 `data/training/plate_segmentation/annotations/instances_reviewed_192.json`에 둡니다. v2 변환 셀은 접시가 없어서 의도적으로 `plate_full`을 누락한 이미지도 `food_visible` 단독 샘플로 학습에 포함합니다.

In [ ]:
# CVAT 검수본 192장을 Drive에 둔 뒤 L4 GPU 런타임에서 실행합니다.
import json, shutil
from pathlib import Path

reviewed = Path('data/training/plate_segmentation/annotations/instances_reviewed_192.json')
images_dir = Path('data/training/plate_segmentation/cvat_images')
output_root = Path('data/training/plate_segmentation/yolo_plate_segmentation_reviewed_192_v2')
run_name = 'yolo11s_plate_seg_reviewed_192_v2_hp'
metrics_path = Path('data/reports/plate_segmenter_metrics_reviewed_192_v2_hp.json')
dataset_yaml = output_root / 'dataset.yaml'
best_weights = Path('runs/plate_segmenter') / run_name / 'weights/best.pt'

assert reviewed.is_file(), f'검수 COCO 파일이 없습니다: {reviewed}'
assert images_dir.is_dir(), f'CVAT 이미지 폴더가 없습니다: {images_dir}'
payload = json.loads(reviewed.read_text(encoding='utf-8'))
assert len(payload.get('images', [])) == 192, f'192장 검수본이 아닙니다: {len(payload.get("images", []))}장'
print('검수 COCO:', reviewed)
print('COCO images:', len(payload.get('images', [])), 'annotations:', len(payload.get('annotations', [])))

# output_root는 학습용 파생 데이터이므로 재실행 때 안전하게 다시 만듭니다.
if output_root.exists():
    shutil.rmtree(output_root)

!python -m scripts.mark_plate_annotation_reviewed --coco-json {reviewed}
!python -m scripts.prepare_plate_segmentation_dataset_v2 --coco-json {reviewed} --images-dir {images_dir} --output-root {output_root}

audit_path = output_root / 'dataset_audit.json'
audit = json.loads(audit_path.read_text(encoding='utf-8'))
print(json.dumps(audit, ensure_ascii=False, indent=2))
assert audit.get('counts', {}).get('train', 0) > 0, audit
assert audit.get('counts', {}).get('val', 0) > 0, audit
assert audit.get('counts', {}).get('test', 0) > 0, audit

!python -m scripts.train_yolo11n_plate_segmenter --data {dataset_yaml} --weights yolo11s-seg.pt --epochs 220 --imgsz 1024 --batch -1 --device 0 --workers 4 --name {run_name} --patience 80 --optimizer AdamW --lr0 0.0015 --lrf 0.01 --warmup-epochs 5 --cos-lr --cache ram --close-mosaic 30 --mosaic 0.5 --mixup 0.03 --copy-paste 0.2 --degrees 7 --translate 0.08 --scale 0.4 --fliplr 0.5 --hsv-s 0.45 --hsv-v 0.3 --overlap-mask
!python -m scripts.evaluate_yolo11n_plate_segmenter --weights {best_weights} --data {dataset_yaml} --imgsz 1024 --device 0 --output {metrics_path}

print(metrics_path.read_text(encoding='utf-8'))
print('best.pt:', best_weights)

# 정량 지표만으로는 plate_full 테두리 완전성을 보장할 수 없으므로 v2 모델의 시각 검토 페이지를 만듭니다.
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import Image as DisplayImage, display
from ultralytics import YOLO

VISUAL_REVIEW_SIZE = 100
visual_review_dir = Path('runs/plate_segmenter_visual_review') / run_name
if visual_review_dir.exists():
    shutil.rmtree(visual_review_dir)
visual_review_dir.mkdir(parents=True, exist_ok=True)

class_colors = {
    0: np.array([255, 140, 0], dtype=np.uint8),
    1: np.array([0, 220, 120], dtype=np.uint8),
}

def _label_path_for(image_path: Path) -> Path:
    split = image_path.parent.name
    return output_root / 'labels' / split / f'{image_path.stem}.txt'

def _label_masks(label_path: Path, shape: tuple[int, int]) -> dict[int, np.ndarray]:
    height, width = shape
    masks = {0: np.zeros((height, width), dtype=np.uint8), 1: np.zeros((height, width), dtype=np.uint8)}
    if not label_path.is_file():
        return masks
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        cls = int(float(parts[0]))
        coords = np.array([float(value) for value in parts[1:]], dtype=np.float32).reshape(-1, 2)
        pts = np.column_stack((coords[:, 0] * width, coords[:, 1] * height)).round().astype(np.int32)
        if cls in masks and len(pts) >= 3:
            cv2.fillPoly(masks[cls], [pts], 255)
    return masks

def _prediction_masks(model: YOLO, image_path: Path, shape: tuple[int, int]) -> dict[int, np.ndarray]:
    height, width = shape
    result = model.predict(source=str(image_path), imgsz=1024, conf=0.25, device=0, verbose=False)[0]
    masks = {0: np.zeros((height, width), dtype=np.uint8), 1: np.zeros((height, width), dtype=np.uint8)}
    if result.masks is None or result.boxes is None:
        return masks
    mask_data = result.masks.data.detach().cpu().numpy()
    classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    for mask, cls in zip(mask_data, classes):
        if cls not in masks:
            continue
        binary = (mask >= 0.5).astype(np.uint8) * 255
        if binary.shape != (height, width):
            binary = cv2.resize(binary, (width, height), interpolation=cv2.INTER_NEAREST)
        masks[cls] = np.maximum(masks[cls], binary)
    return masks

def _overlay(image_rgb: np.ndarray, masks: dict[int, np.ndarray], alpha: float = 0.45) -> np.ndarray:
    output = image_rgb.copy()
    for cls, mask in masks.items():
        region = mask > 0
        if np.any(region):
            output[region] = (output[region].astype(np.float32) * (1.0 - alpha) + class_colors[cls].astype(np.float32) * alpha).astype(np.uint8)
    return output

review_candidates = []
for split in ('test', 'val', 'train'):
    review_candidates.extend(sorted((output_root / 'images' / split).glob('*')))
review_candidates = [path for path in review_candidates if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
random.Random(42).shuffle(review_candidates)
review_images = review_candidates[:min(VISUAL_REVIEW_SIZE, len(review_candidates))]
review_model = YOLO(str(best_weights))
saved_pages = []
for index, image_path in enumerate(review_images, start=1):
    image_rgb = np.array(PILImage.open(image_path).convert('RGB'))
    height, width = image_rgb.shape[:2]
    gt_masks = _label_masks(_label_path_for(image_path), (height, width))
    pred_masks = _prediction_masks(review_model, image_path, (height, width))
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    panels = [('original', image_rgb), ('ground truth', _overlay(image_rgb, gt_masks)), ('v2 prediction', _overlay(image_rgb, pred_masks))]
    for ax, (title, panel) in zip(axes, panels):
        ax.imshow(panel)
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'{index:03d}. {image_path.name} | orange=plate_full, green=food_visible')
    fig.tight_layout()
    page_path = visual_review_dir / f'visual_review_{index:03d}_{image_path.stem}.jpg'
    fig.savefig(page_path, dpi=140)
    plt.close(fig)
    saved_pages.append(page_path)

for page_path in saved_pages[:12]:
    display(DisplayImage(filename=str(page_path)))
print('visual review pages:', len(saved_pages), visual_review_dir)